# LLM 2일차 (3) — AI 에이전트와 도구 : Tavily 웹 검색 · `create_agent`

수업 필기 `LangChain_실습4` · `LangChain_실습5` 를 정리 · 개선한 노트입니다.
(교재 `AI Agent` 교재10 **p.27~33** — 에이전트 · 도구(이름 · 설명 · 입력 스키마) · TavilySearch 매개변수 · 검색 실습 · 실시간 검색 에이전트)

```
사용자 질문 → [LLM] 도구가 필요한가?
                 ├─ 아니오 → 답변
                 └─ 예 → tool_calls(이름 · 인자) → [도구 실행] → ToolMessage(결과) → 다시 [LLM] → …
```

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다(Day 046 과 같은 방식).
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | API 대신 **정해 둔 답을 내는 가짜 LLM** 으로 체인 · 에이전트의 **흐름**만 실제로 돌린 셀 (답 내용은 의미 없음) |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 (프롬프트 채우기 · 토큰 세기 · 도구 정보 · 기록 분석) |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langchain` · `langgraph` | `create_agent` (내부적으로 LangGraph 그래프) | `pip install langchain` |
| `langchain-openai` | `ChatOpenAI` | `pip install langchain-openai` |
| `langchain-tavily` | `TavilySearch` 웹 검색 도구 | `pip install langchain-tavily` |
| `python-dotenv` | API 키 | `pip install python-dotenv` |

**API 키 2개** : `.env` 에 `OPENAI_API_KEY="sk-…"` · `TAVILY_API_KEY="tvly-…"` ([tavily.com](https://tavily.com) 가입 후 발급, 무료 월 1,000 크레딧). **실행** : 맨 위부터 모두 실행. 기록 분석(§4)은 이 폴더의 필기 원본 `LangChain_실습4.ipynb` · `LangChain_실습5.ipynb` 를 읽는다.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langchain langchain-openai langchain-tavily python-dotenv

In [2]:
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
import os
from dotenv import load_dotenv, find_dotenv
loaded = load_dotenv(find_dotenv(usecwd=True))      # 지금 폴더에서 위로 올라가며 .env 를 찾는다
print('.env 읽음 :', loaded)
print('OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')), '| TAVILY_API_KEY 있음 :', bool(os.getenv('TAVILY_API_KEY')))

.env 읽음 : False
OPENAI_API_KEY 있음 : False | TAVILY_API_KEY 있음 : False


## 1. 도구의 세 가지 구성 요소 — 이름 · 설명 · 입력 스키마 (교재 p.28)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

search_tool = TavilySearch(max_results=3)            # 키는 .env 의 TAVILY_API_KEY 를 자동으로 읽는다
print(search_tool)
print(search_tool.name)
print(search_tool.description)

max_results=3 api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None)
tavily_search
A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.


In [5]:
# 키 없이도 볼 수 있다 — 도구 클래스에 정해진 ① 이름 ② 설명 ③ 입력 스키마 (LLM 이 보는 정보 전부)
fields = TavilySearch.model_fields
schema = fields['args_schema'].default.model_json_schema()
print('① 이름 :', fields['name'].default)
print('② 설명 :', fields['description'].default[:120], '…')
print('③ 입력 스키마 (LLM 이 채울 수 있는 인자) :')
for k, v in schema['properties'].items():
    choices = v.get('enum') or [e for a in v.get('anyOf', []) for e in a.get('enum', [])]
    print(f"   {k:16s} {'(필수)' if k in schema.get('required', []) else '      '} {choices if choices else ''}")

① 이름 : tavily_search
② 설명 : A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions …
③ 입력 스키마 (LLM 이 채울 수 있는 인자) :
   query            (필수) 
   include_domains         
   exclude_domains         
   search_depth            ['basic', 'advanced', 'fast', 'ultra-fast']
   include_images          
   time_range              ['day', 'week', 'month', 'year']
   topic                   ['general', 'news', 'finance']
   start_date              
   end_date                


- `max_results=3` 처럼 **만들 때 정한 값**은 고정이고, `query` · `search_depth` · `time_range` · `topic` 같은 **입력 스키마의 칸은 LLM 이 호출할 때마다 정한다** — 수업 에이전트도 스스로 `search_depth='advanced'` · `time_range='year'` 를 골랐다(§4).
- LLM 은 이 **이름 · 설명 · 스키마만 보고** 언제 어떤 도구를 쓸지 정한다. 설명을 잘 쓰는 것이 곧 도구 사용 품질이다.

> **📝 고친 부분 (교재 p.29)** `search_depth` 를 `'basic'` · `'advanced'(권장)` 두 가지로 적었지만, 설치된 `langchain-tavily 0.2.18` 의 입력 스키마는 위처럼 **`'fast'` · `'ultra-fast'`** 도 받는다. 그리고 `'advanced'` 는 검색 1번에 **2크레딧**(`'basic'` 은 1크레딧, 무료 플랜 월 1,000크레딧)이라 무조건 '권장' 은 아니다 — 수업 에이전트는 검색 3번 × advanced = **6크레딧**을 썼다. 교재 p.28 '외부 시스템(…)**와**' → '**과**'.

In [6]:
# 내 함수도 도구가 된다 — 함수 이름 = 도구 이름, docstring = 설명, 타입 힌트 = 입력 스키마
from langchain_core.tools import tool

@tool
def won_to_dollar(won: int, rate: float = 1400.0) -> str:
    """원화 금액을 달러로 바꾼다. 환율(rate)을 주지 않으면 1달러 = 1400원으로 계산한다."""
    return f'{won:,}원 = ${won / rate:,.2f}'

print(won_to_dollar.name, '|', won_to_dollar.description)
print(won_to_dollar.args)
print(won_to_dollar.invoke({'won': 2_800_000}))

won_to_dollar | 원화 금액을 달러로 바꾼다. 환율(rate)을 주지 않으면 1달러 = 1400원으로 계산한다.
{'won': {'title': 'Won', 'type': 'integer'}, 'rate': {'default': 1400.0, 'title': 'Rate', 'type': 'number'}}
2,800,000원 = $2,000.00


## 2. 검색 실행 (필기 `LangChain_실습4`, 교재 p.29~30)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

query = 'Tesla의 2026년 2분기 실적'
query_results = search_tool.invoke(query)
print(query_results)

{'query': 'Tesla의 2026년 2분기 실적', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.cmelitegroup.com/ko/knowledge-hub/tesla-earnings-date', 'title': '테슬라 실적 발표일: 2026년 4분기 발표 시간 및 예상치 | CapitalMEG', 'content': '## 전 분기: 2026년 2분기\n\n7월 22일 발표된 테슬라의 2026년 2분기 실적 보고서에 따르면, 매출은 282억 달러로, 애널리스트들이 예상한 약 264억 달러를 상회했다. 조정 주당 순이익은 0.33달러로, 0.50달러에 가까운 예상치를 밑돌았다. \n\n실적 부진이 매출 호조를 상쇄했다. 다음 거래일 주가는 약 14.5% 하락했다.\n\n3분기 실적 발표를 앞두고 테슬라는 해당 분기 동안 486,532대의 차량을 인도하고 464,391대를 생산했으며, 13.7 GWh 규모의 에너지 저장 시스템을 구축했다고 밝혔다. \n\n## 3분기를 앞두고 트레이더들이 주목한 5가지 [...] ## 참고 문헌\n\n 테슬라, 2026년 3분기 차량 생산량, 인도량 및 실적 발표일 공지 (양식 8-K, 첨부 문서 99.1), 2026년 10월. \n\n 테슬라, 2026년 2분기 실적, 2026년 7월 22일 발표; PrimeXBT가 보도한 추정치. \n\n PrimeXBT를 통해 제공된 Zacks 컨센서스, 2026년 10월 14일 기준. \n\n MarketBeat, 테슬라 실적, 2026년 10월 21일 확인. \n\n 테슬라, “테슬라, 2026년 3분기 실적 발표”, 2026년 10월 21일 (Stock Titan 인용). \n\n Teslarati, “테슬라 2026년 3분기 실적: 매출은 예상치와 일치, 주당순이익(EPS)은 예상치를 소폭 상회”, 2026년 10월 21일. [...] 영업이익: 7억 1,

In [8]:
# 수업 검색 결과(📼)를 다시 읽어 정리 — 결과는 dict 이므로 출처 · 점수 · 내용 길이를 표로
import ast, json as _json, pandas as pd
rec4 = _json.load(open('LangChain_실습4.ipynb'))['cells'][9]['outputs'][0]['text']
results = ast.literal_eval(''.join(rec4))                       # print 된 dict 글자 → dict
table = pd.DataFrame([{'점수': r['score'], '출처': r['url'][:55], '제목': r['title'][:30], '내용 글자 수': len(r['content'])}
                      for r in results['results']])
print('응답 키 :', list(results.keys()), '| 걸린 시간', results['response_time'], '초')
table

응답 키 : ['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id'] | 걸린 시간 2.93 초


,점수,출처,제목,내용 글자 수
0,0.794385,https://www.cmelitegroup.com/ko/knowledge-hub/...,테슬라 실적 발표일: 2026년 4분기 발표 시간 및,1126
1,0.701713,https://assets-ir.tesla.com/tesla-contents/IR/...,[PDF] Q2 2026 Update | Tesla,1352
2,0.547646,https://www.hankyung.com/article/202607230480i,"테슬라, 2분기 매출 26%↑…사상 최대 실적에도 영업",850


In [9]:
# 세 결과가 말하는 2분기 매출 · EPS 가 서로 맞는지 — 공식 IR(PDF)과 비교
import re
for r in results['results']:
    c = r['content']
    rev = re.findall(r'Total revenues[^\n]*?28,236|매출은 \d+억 달러', c)
    eps = re.findall(r'조정 주당 순이익은 [\d.]+달러|\(non-GAAP\) [\d. ]+0\.33', c)
    paywall = '구독' in c and '무단 전재' in c
    print(f"{r['url'][:45]:46s} 매출 {rev[:1]} · EPS {eps[:1]} · 유료 안내문 {paywall}")

https://www.cmelitegroup.com/ko/knowledge-hub  매출 ['매출은 282억 달러'] · EPS ['조정 주당 순이익은 0.33달러'] · 유료 안내문 False
https://assets-ir.tesla.com/tesla-contents/IR  매출 ['Total revenues 22,496 28,095 24,901 22,387 28,236'] · EPS ['(non-GAAP) 0.40 0.50 0.50 0.41 0.33'] · 유료 안내문 False
https://www.hankyung.com/article/202607230480  매출 [] · EPS [] · 유료 안내문 True


- 공식 IR 자료(PDF)의 2분기 매출 **28,236 백만 달러**와 블로그의 '**282억 달러**', non-GAAP EPS **0.33** 이 서로 맞는다.
- 세 번째 결과(신문 기사)의 내용은 **유료 구독 안내문**뿐이다 — 검색 '점수'(0.55)가 낮은 결과는 쓸모없는 글일 수 있다. 에이전트에게 넘기기 전에 출처 · 내용을 확인해야 하는 이유.

> **📝 고친 부분 (원본 `LangChain_실습4`)** 결과를 `print(dict)` 로 한 줄에 통째로 출력 → 출처 · 점수 · 길이 표로 보고, 숫자는 공식 출처와 대조.

## 3. 에이전트 — `create_agent` (필기 `LangChain_실습5`, 교재 p.31~33)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

search_tool = TavilySearch(max_results=3, search_depth='advanced')
tools = [search_tool]
print(f'도구 목록: {tools}')

도구 목록: [TavilySearch(search_depth='advanced', max_results=3, api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]


In [11]:
system_prompt = """
너는 웹 검색 도구를 적극 활용하여 사용자 질문에 정확하고 객관적인 답변을 제공하는 전문 AI 에이전트야.
전달받은 웹 검색 결과에 기반하여 환각(Hallucination) 없이 사실만을 정리해 답변해야 해.
그리고 답변은 반드시 한국어로 작성할 것
"""
print(system_prompt.strip())

너는 웹 검색 도구를 적극 활용하여 사용자 질문에 정확하고 객관적인 답변을 제공하는 전문 AI 에이전트야.
전달받은 웹 검색 결과에 기반하여 환각(Hallucination) 없이 사실만을 정리해 답변해야 해.
그리고 답변은 반드시 한국어로 작성할 것


In [12]:
# 🧪 가짜 LLM : 정해 둔 메시지를 차례로 돌려준다 (API · 인터넷 · 요금 없음)
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

class ScriptedLLM(GenericFakeChatModel):
    """create_agent 도 쓸 수 있게 bind_tools 만 흉내 낸 가짜 모델"""
    def bind_tools(self, tools, **kwargs):
        return self

def fake_llm(*replies):
    return ScriptedLLM(messages=iter([r if isinstance(r, AIMessage) else AIMessage(content=r) for r in replies]))

In [13]:
# 🧪 모의 실행 — API 없이 정해 둔 답을 내는 가짜 LLM 으로 '흐름' 만 확인

# 에이전트의 '흐름' 만 확인 — 가짜 LLM 이 ① 검색 요청 → ② 최종 답 을 차례로 내고, 도구는 오프라인 가짜 검색
from langchain_core.tools import tool

@tool
def tavily_search(query: str, search_depth: str = 'basic') -> str:
    """(오프라인 모의) 웹 검색 결과를 돌려준다."""
    return _json.dumps({'query': query, 'results': [{'title': '가짜 결과', 'content': '…'}]}, ensure_ascii=False)

script = fake_llm(
    AIMessage(content='', tool_calls=[{'name': 'tavily_search', 'args': {'query': '2026 generative AI market trends', 'search_depth': 'advanced'},
                                      'id': 'call_1', 'type': 'tool_call'}]),
    '검색 결과를 바탕으로 한 최종 답변(모의)입니다.')
demo_agent = create_agent(model=script, tools=[tavily_search], system_prompt=system_prompt)
out = demo_agent.invoke({'messages': [{'role': 'user', 'content': '2026년 생성형 AI 시장 트렌드를 정리해 줘'}]})
for m in out['messages']:
    extra = m.tool_calls if getattr(m, 'tool_calls', None) else ''
    print(f'{type(m).__name__:12s} {m.content[:60]!r} {extra}')

HumanMessage '2026년 생성형 AI 시장 트렌드를 정리해 줘' 
AIMessage    '' [{'name': 'tavily_search', 'args': {'query': '2026 generative AI market trends', 'search_depth': 'advanced'}, 'id': 'call_1', 'type': 'tool_call'}]
ToolMessage  '{"query": "2026 generative AI market trends", "results": [{"' 
AIMessage    '검색 결과를 바탕으로 한 최종 답변(모의)입니다.' 


In [14]:
# 에이전트 = 그래프 : model ⇄ tools 를 '도구 호출이 없을 때까지' 반복
print(demo_agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools<hr/><small><em>__tools = tavily_search</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

llm = ChatOpenAI(model='gpt-5-nano')
agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)
response = agent.invoke({
    'messages': [{'role': 'user', 'content': '2026년 생성형 AI 시장 트렌드 및 최신 AI 에이전트 프레임워크 동향을 간단하게 정리해 줘'}]
})
text = response['messages'][-1].content            # 마지막 메시지 = 최종 답변
print(text)

다음은 웹 결과를 바탕으로 한 2026년 생성형 AI 시장 트렌드와 최신 AI 에이전트 프레임워크 동향의 간단 요약입니다.

1) 2026년 생성형 AI 시장 트렌드
- 시장 규모와 성장성
  - 글로벌 생성형 AI 시장은 2025년 약 222억 달러에서 2026년 약 296억 달러로 증가 전망. 2026–2033년 CAGR 약 40%대(CAGR 40.8%로 제시, Grand View Research).
  - 추가로 2024–2030년 CAGR 약 46%대 예측도 언급되는 등 빠른 성장세 지속 전망.
- 채택 현황과 산업 수요
  - 글로벌 대기업의 광범위한 도입 흐름: Fortune 500 기업의 약 92%가 Gen AI를 채택했다는 보도 포함.
  - 주요 수요 산업으로는 소비자 서비스, 금융, 헬Healthcare 등에서 성장 기회 확대 언급.
  - 기업들은 가상 비서/챗봇 외에 음성-멀티모달 솔루션, 인사이트 자동화 등 적용 분야 확대.
- 기술/제품 트렌드
  - 생산형 에이전트(production agents)와 루틴 작업 자동화를 위한 저가 소형 모델의 활용 증가.
  - 멀티모달 입력, 도구 호출을 위한 Stateless MCP 등 도구 연결 방식의 진화.
  - 맞춤형 평가(Evals) 체계 및 규제 대응 강화(EU AI Act 준수 등) 필요성 증가.
- 비용과 거버넌스
  - 추론 비용이 높은 것이 주요 확산 장애 요인으로 지목되며, 비용 모니터링과 제어가 중요해짐(일부 자료에서 49%가 비용 이슈를 문제로 제시).
- 에코시스템/모델 정합성
  - GPT-6 Astra, Gemini 4 Argon 등 차세대 대형 모델의 출시 소식이 시장 기대를 뒷받침.
  - 텍스트-이미지-비디오 등 멀티모달 기능과 고도화된 평가/거버넌스 도구의 필요성 증가.

참고: Future AI 요약, Grand View Research의 시장 규모/성장성, Master of Code의 2026년 AI 트렌드 및 채택 통계 등에서 확인되는 내용입니다.

2) 

> **📝 고친 부분 (원본 cell 15)** `print(f'[최종 에이전트 답변]: {response}')` 는 답변이 아니라 **메시지 전체(질문 · 도구 호출 · 검색 결과 원문 · 토큰 정보)** 를 2만 2천 글자로 출력한다 → 최종 답은 `response['messages'][-1].content`, 과정은 아래처럼 메시지 종류별로 요약해서 본다.

## 4. 수업 실행 기록 분석 — 에이전트는 무엇을 했나

In [16]:
rec5 = ''.join(_json.load(open('LangChain_실습5.ipynb'))['cells'][15]['outputs'][0]['text'])
final = ''.join(_json.load(open('LangChain_실습5.ipynb'))['cells'][16]['outputs'][0]['text'])
kinds = re.findall(r'(HumanMessage|AIMessage|ToolMessage)\(content=', rec5)
calls = re.findall(r"\{'name': 'tavily_search', 'args': (\{[^}]*\})", rec5)
usage = re.findall(r"'completion_tokens': (\d+), 'prompt_tokens': (\d+).*?'reasoning_tokens': (\d+)", rec5)
finish = re.findall(r"'finish_reason': '(\w+)'", rec5)
print('메시지 순서 :', ' → '.join(kinds))
print('도구 호출', len(calls), '번 (한 번에 나란히) :')
for c in calls:
    print('   ', ast.literal_eval(c))
for i, ((out_t, in_t, rea), fr) in enumerate(zip(usage, finish), 1):
    print(f'LLM 호출 {i} : 입력 {int(in_t):,} · 출력 {int(out_t):,} (그중 추론 {int(rea):,}) · 끝난 이유 {fr}')
urls = re.findall(r'"url": "([^"]+)"', rec5)
print('검색 결과 출처', len(urls), '개 (검색 3번 × 3개) :')
for u in urls:
    print('   ', u[:80])

메시지 순서 : HumanMessage → AIMessage → ToolMessage → ToolMessage → ToolMessage → AIMessage
도구 호출 3 번 (한 번에 나란히) :
    {'query': '2026 generative AI market trends', 'search_depth': 'advanced', 'time_range': 'year', 'include_images': False}
    {'query': 'AI agent framework trends 2026', 'search_depth': 'advanced', 'time_range': 'year', 'include_images': False}
    {'query': 'LangChain 2026 updates agent framework', 'search_depth': 'advanced', 'time_range': 'year', 'include_images': False}
LLM 호출 1 : 입력 1,376 · 출력 713 (그중 추론 576) · 끝난 이유 tool_calls
LLM 호출 2 : 입력 5,468 · 출력 4,174 (그중 추론 2,816) · 끝난 이유 stop
검색 결과 출처 9 개 (검색 3번 × 3개) :
    https://futureagi.com/blog/generative-ai-trends-2026
    https://www.grandviewresearch.com/industry-analysis/generative-ai-market-report
    https://masterofcode.com/blog/generative-ai-statistics
    https://pub.towardsai.net/top-ai-agent-frameworks-in-2026-a-production-ready-com
    https://www.langchain.com/resources/ai-agent-frameworks
    https://www.lin

In [17]:
# 최종 답의 숫자 · 고유명사가 검색 결과(ToolMessage)에 실제로 있었나
tool_text = ' '.join(re.findall(r'ToolMessage\(content=(.*?), name=', rec5, flags=re.S))
# (최종 답의 표현, 검색 결과에서 찾을 표현) — 영어 원문을 한국어로 옮긴 것은 원문 표현으로 찾는다
claims = [('222억 달러', 'USD 22.2 billion'), ('296억 달러', 'USD 29.6 billion'), ('40.8%', '40.8%'), ('92%', '92%'),
          ('49%', '49%'), ('GPT-6 Astra', 'GPT-6 Astra'), ('Gemini 4 Argon', 'Gemini 4 Argon'),
          ('Stateless MCP', 'stateless MCP'), ('80% 케이스', '80% case'), ('LangChain 1.0', 'LangChain 1.0'), ('CrewAI', 'CrewAI')]
for ans, src in claims:
    print(f'{ans:16s} 최종 답에 {ans in final!s:5s} | 검색 결과에 {src in tool_text!s:5s} ← {src!r}')
i = tool_text.find('USD 22.2 billion')
print('\n원문 :', tool_text[i - 51:i + 130].replace('\\n', ' '))
j = tool_text.find('80% case')
print('원문 :', tool_text[j - 59:j + 170].replace('\\n', ' '))
k = final.find('80% 케이스')
print('답변 :', final[k - 30:k + 75])

222억 달러          최종 답에 True  | 검색 결과에 True  ← 'USD 22.2 billion'
296억 달러          최종 답에 True  | 검색 결과에 True  ← 'USD 29.6 billion'
40.8%            최종 답에 True  | 검색 결과에 True  ← '40.8%'
92%              최종 답에 True  | 검색 결과에 True  ← '92%'
49%              최종 답에 True  | 검색 결과에 True  ← '49%'
GPT-6 Astra      최종 답에 True  | 검색 결과에 True  ← 'GPT-6 Astra'
Gemini 4 Argon   최종 답에 True  | 검색 결과에 True  ← 'Gemini 4 Argon'
Stateless MCP    최종 답에 True  | 검색 결과에 True  ← 'stateless MCP'
80% 케이스          최종 답에 True  | 검색 결과에 True  ← '80% case'
LangChain 1.0    최종 답에 True  | 검색 결과에 True  ← 'LangChain 1.0'
CrewAI           최종 답에 True  | 검색 결과에 True  ← 'CrewAI'

원문 : The global generative AI market size was valued at USD 22.2 billion in 2025 and is projected to grow from USD 29.6 billion in 2026 to USD 324.7 billion by 2033, at a CAGR of 40.8% f
원문 : The practical rule: use LangChain’s high-level API for the 80% case, chain composition, RAG, and simple agents. Drop to LangGraph directly for multi-agent coord

- 이 에이전트는 LLM 을 **두 번** 불렀다 — ① 질문을 보고 검색어 3개를 **한 번에(병렬)** 요청 → 도구 3번 실행 → ② 검색 결과 3개를 모두 읽고 최종 답. 두 번째 호출의 입력이 첫 호출보다 훨씬 크다(검색 결과 원문이 들어가서).
- 최종 답의 숫자 · 이름은 **모두 검색 결과에 있던 것**이다(위 표). `USD 29.6 billion` → '**296억 달러**' 처럼 단위를 바꿔 옮긴 것도 맞게 바꿨다(29.6 billion = 296억). 다만 `GPT-6 Astra` · `Gemini 4 Argon` 은 한 블로그(futureagi) 글에만 나온 이름이라 **출처가 믿을 만한지는 따로 확인**해야 한다 — 시스템 프롬프트의 '환각 없이' 는 '검색 결과에 없는 말을 지어내지 마' 이지, 검색 결과가 사실이라는 보장이 아니다.
- 그런데 **단어가 있다고 뜻까지 맞는 것은 아니다.** 원문은 '80% 는 LangChain 고수준 API, **멀티 에이전트 · 상태 분기 · 사람 개입은 LangGraph 로 직접 내려가라**(*Drop to LangGraph directly*)' 인데, 답변은 '복잡한 상태 관리가 필요할 때 **LangGraph 로 내려가지 않고도** 제어 가능' 으로 **뜻을 거꾸로** 옮겼다. 같은 답의 뒷부분('고도 제어가 필요하면 LangGraph 로 직접 진입')과도 모순된다.
- 최종 답의 '`헬Healthcare`'(원문 *healthcare* 를 옮기다 섞임) 같은 깨진 표현도 그대로 남았다 — 결과는 사람이 검토해야 한다.

## 5. 교재 방식 — 체인 2개로 직접 만드는 에이전트 (교재 p.31~32)

`create_agent` 가 안에서 반복해 주는 일을 교재는 **체인 2개**로 직접 만든다 : ① `tool_calling_chain`(도구가 필요한지 판단 · 도구 호출 생성) → 도구 실행 → `ToolMessage` 를 기록에 추가 → ② `final_answer_chain`(최종 글).

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, ToolMessage

agent_prompt = ChatPromptTemplate.from_messages([('system', system_prompt), MessagesPlaceholder('messages')])

def run_two_chain_agent(llm, tool_list, question):
    tool_map = {t.name: t for t in tool_list}
    tool_calling_chain = agent_prompt | llm.bind_tools(tool_list)        # 1차 체인 : 도구 호출 판단
    final_answer_chain = agent_prompt | llm | StrOutputParser()            # 2차 체인 : 최종 답
    messages = [HumanMessage(question)]
    ai = tool_calling_chain.invoke({'messages': messages})
    messages.append(ai)
    if not ai.tool_calls:                                                  # 도구가 필요 없으면 바로 답
        return ai.content, messages
    for call in ai.tool_calls:                                             # 도구 실행 → 결과를 기록에 추가
        result = tool_map[call['name']].invoke(call['args'])
        messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
    return final_answer_chain.invoke({'messages': messages}), messages

In [19]:
# 🧪 모의 실행 — API 없이 정해 둔 답을 내는 가짜 LLM 으로 '흐름' 만 확인

two_llm = fake_llm(AIMessage(content='', tool_calls=[{'name': 'won_to_dollar', 'args': {'won': 2800000}, 'id': 'c1', 'type': 'tool_call'}]),
                   '280만 원은 약 2,000달러입니다.')
answer, msgs = run_two_chain_agent(two_llm, [won_to_dollar], '280만 원은 몇 달러야?')
for m in msgs:
    print(f'{type(m).__name__:12s} {m.content!r:45.45s} {getattr(m, "tool_calls", "") or ""}')
print('최종 :', answer)

HumanMessage '280만 원은 몇 달러야?'                              
AIMessage    ''                                            [{'name': 'won_to_dollar', 'args': {'won': 2800000}, 'id': 'c1', 'type': 'tool_call'}]
ToolMessage  '2,800,000원 = $2,000.00'                      
최종 : 280만 원은 약 2,000달러입니다.


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

answer, msgs = run_two_chain_agent(ChatOpenAI(model='gpt-5-nano'), [TavilySearch(max_results=3), won_to_dollar],
                                   '테슬라 2026년 2분기 매출을 원화로 바꾸면 대략 얼마야?')
print(answer)

> **📝 고친 부분 (교재 p.31 과 필기의 차이)** 교재는 '`tool_calling_chain` · `final_answer_chain` 2개의 체인' 으로 설명하지만 필기는 LangChain 1.0 의 **`create_agent`** 를 썼다. 체인 2개 방식은 도구를 **한 번만** 쓰고 끝나고, `create_agent` 는 LLM 이 '더 이상 도구가 필요 없다' 고 할 때까지 **반복**한다(ReAct). 위 두 방식을 모두 실었다.

## 정리

| 할 일 | 코드 |
|---|---|
| 도구 만들기 | `TavilySearch(max_results=3, search_depth='advanced')` · `@tool` 함수 |
| 도구 정보 | `.name` · `.description` · `.args` (LLM 이 보는 것) |
| 에이전트 | `create_agent(model=llm, tools=[…], system_prompt=…)` |
| 실행 | `agent.invoke({'messages': [{'role': 'user', 'content': 질문}]})` |
| 최종 답 | `response['messages'][-1].content` |
| 과정 보기 | 메시지 종류 · `tool_calls` · `usage_metadata` |